# GraphAgents: Knowledge Graph-Guided Agentic AI for Cross-Domain Materials Design

#### Authors: Isabella Stewart, Tarjei Hage, Yu-Chuan (Michael) Hsu, and Markus J. Buehler, MIT, 2025 
#### Corresponding author: Markus J. Buehler, mbuehler@MIT.EDU
#### LAMM, Massachusetts Institute of Technology


## Environment Initialization

In [ ]:
import os
import sys
import re
import glob
import uuid
import shutil
import hashlib
import importlib
from typing import Optional, Union, Tuple, List, Dict, Any, Literal, TypeVar, cast, Callable

import autogen, openai
import numpy as np
import numpy.typing as npt
import pandas as pd
import networkx as nx
import torch  # Used for GPU tensor operations, pooling, and device transfers.
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from IPython import get_ipython

# ChromaDB / Vector DB interfaces
from chromadb.api.types import EmbeddingFunction, Embeddings
from chromadb import PersistentClient
from chromadb.config import Settings

# AutoGen contrib + utils
from autogen.agentchat import UserProxyAgent
from autogen.agentchat.agent import Agent
from autogen.agentchat.contrib.vectordb.base import Document, QueryResults, VectorDB, VectorDBFactory
from autogen.agentchat.contrib.vectordb.utils import (
    chroma_results_to_query_results,
    filter_results_by_distance,
    get_logger,
)
from autogen.agentchat.contrib.vectordb.chromadb import ChromaVectorDB
from autogen.code_utils import extract_code
from autogen.token_count_utils import count_token
from autogen.formatting_utils import colored

# Notebook export utilities
import nbformat
from nbconvert import HTMLExporter


In [ ]:
# !curl http://localhost:<port>/v1/models
# NOTE: Optional for endpoint sanity check.

In [ ]:
# sys.path.insert(0, '../GraphReasoning')
# NOTE: If not pip installing GraphReasoning 

## LLM API Configuration

In [ ]:
# LLM api configuration

base_url= "http://localhost:8081/v1"
api_key = "NULL"

config_list = [
    {
        "model": "Llama3.3",
        "base_url": base_url,
        "api_key": api_key,
        "max_tokens": 40000
    },
]

llm_config = {
    "cache_seed": 9527,         # Seed for caching and reproducibility
    "config_list": config_list, # A list of OpenAI-compatible API configurations
    "temperature": 0,           # Deterministic sampling for scientific runs
    "max_tokens": 40000,
    "timeout": 1200,            # Timeout (seconds)
}

## Graph Dataset Configuration

In [ ]:
# symlinks to directories of material properties and PFAS graph, embedding files (.pkl), etc.
data_dir_materialproperties = './GRAPHDATA_MATERIALPROPERTIES'
data_dir_output_materialproperties= './GRAPHDATA_OUTPUT_MATERIALPROPERTIES'
embedding_file_materialproperties = 'SG_LLAMA4_70b.pkl'

data_dir_PFAS = './GRAPHDATA_PFAS'
data_dir_output_PFAS = './GRAPHDATA_OUTPUT_PFAS'
embedding_file_PFAS = 'SG_LLAMA4_70b.pkl'

max_tokens = config_list[0]['max_tokens']

## Embedding and Token Configuration

In [ ]:
# Define embedding tokenizer/model used across both VectorDB and Graph retrieval components.

embedding_tokenizer = ''  # Placeholder for the other embedding models that need tokenizer
embedding_model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)

In [ ]:
# Token counting to estimate context budgets when assembling retrieval contexts.

def custom_token_count_function(text, placeholder=''):
    inputs = embedding_model.tokenizer(text, return_tensors='pt', truncation=True)
    return len(inputs["input_ids"][0])

In [ ]:
# ChromaDB-compatible embedding function wrapper
# Provides normalization and dual-mode encoding (HF-style or SentenceTransformer).

Embeddable = Union[str, nx.DiGraph]
D = TypeVar("D", bound=Embeddable, contravariant=True)

class TransformerEmbeddingFunction(EmbeddingFunction[D]):
    def __init__(
        self,
        embedding_tokenizer,
        embedding_model,
        cache_dir: Optional[str] = None,
    ):
        try:
            from transformers import AutoModel, AutoTokenizer  # noqa: F401 (import retained)
            self._torch = importlib.import_module("torch")
            # Use provided tokenizer/model; HF constructors intentionally not invoked here.
            self._tokenizer = embedding_tokenizer
            self._model = embedding_model
        except ImportError:
            raise ValueError(
                "The transformers and/or pytorch package is not installed. Please install it with "
                "pip install transformers or pip install torch"
            )

    @staticmethod
    def _normalize(vector: npt.NDArray) -> npt.NDArray:
        """L2-normalize embedding vectors to unit length for cosine similarity."""
        norm = np.linalg.norm(vector)
        if norm == 0:
            return vector
        return vector / norm

    def __call__(self, input: D) -> Embeddings:
        """Tokenize and embed input; return normalized embeddings as Python lists."""
        if self._tokenizer:
            inputs = self._tokenizer(
                input, padding=True, truncation=True, return_tensors="pt"
            ).to('cuda:0')
            outputs = self._model(**inputs)
            try:
                embeddings = outputs.last_hidden_state.mean(dim=1).detach().numpy()
            except:
                embeddings = outputs.hidden_states[-1].mean(dim=1).detach().to(torch.float).cpu().numpy()
        else:
            embeddings = self._model.encode(input)
        return [e.tolist() for e in self._normalize(embeddings)]

In [ ]:
embedding_function = TransformerEmbeddingFunction(embedding_tokenizer=embedding_tokenizer, embedding_model=embedding_model)

## Load PFAS Material Properties Knowledge Graph and Generate Node Embeddings

In [ ]:
# Load PFAS Material Properties Knowledge Graph and their corresponding node embeddings.

G_materialproperties = nx.read_graphml(f'{data_dir_output_materialproperties}/simple_graph_graphML_simplified.graphml')
relation = nx.get_edge_attributes(G_materialproperties, "title")  
nx.set_edge_attributes(G_materialproperties, relation, "relation")
nx.set_node_attributes(G_materialproperties, nx.pagerank(G_materialproperties), "pr") 
print(f'KG loaded: {G_materialproperties}')

from GraphReasoning import load_embeddings, generate_node_embeddings, save_embeddings
generate_new_embeddings = False # do not remake if already made

if os.path.exists(f'{data_dir_materialproperties}/{embedding_file_materialproperties}'):
    generate_new_embeddings = False

if generate_new_embeddings:
    try:
        node_embeddings_materialproperties = generate_node_embeddings(G_materialproperties, embedding_tokenizer, embedding_model, )
    except:
        node_embeddings_materialproperties = generate_node_embeddings(nx.DiGraph(), embedding_tokenizer, embedding_model, )
        save_embeddings(node_embeddings_materialproperties, f'{data_dir_materialproperties}/{embedding_file_materialproperties}')
else:
    filename = f"{data_dir_materialproperties}/{embedding_file_materialproperties}"
    node_embeddings_materialproperties = load_embeddings(f'{data_dir_materialproperties}/{embedding_file_materialproperties}')

## Load PFAS Knowledge Graph and Generate Node Embeddings

In [ ]:
# Load PFAS Knowledge Graph from full-text articles and generate node embeddings

G_PFAS = nx.read_graphml(f'{data_dir_output_PFAS}/simple_graph_graphML_simplified.graphml')
relation = nx.get_edge_attributes(G_PFAS, "title")
nx.set_edge_attributes(G_PFAS, relation, "relation")
nx.set_node_attributes(G_PFAS, nx.pagerank(G_PFAS), "pr")
print(f'KG loaded: {G_PFAS}')

from GraphReasoning import load_embeddings, generate_node_embeddings, save_embeddings
generate_new_embeddings = False  #  do not remake if already made

if os.path.exists(f'{data_dir_PFAS}/{embedding_file_PFAS}'):
    generate_new_embeddings = False

if generate_new_embeddings:
    try:
        node_embeddings_PFAS = generate_node_embeddings(G_PFAS, embedding_tokenizer, embedding_model, )
    except:
        node_embeddings_PFAS = generate_node_embeddings(nx.DiGraph(), embedding_tokenizer, embedding_model, )
        save_embeddings(node_embeddings_PFAS, f'{data_dir_PFAS}/{embedding_file_PFAS}')
else:
    filename = f"{data_dir_PFAS}/{embedding_file_PFAS}"
    node_embeddings_PFAS = load_embeddings(f'{data_dir_PFAS}/{embedding_file_PFAS}')

## Alternatively Load Existing Embeddings if Already Generated

In [ ]:
# Directly loading existing node embeddings with load_embeddings

from GraphReasoning import load_embeddings
node_embeddings_materialproperties = load_embeddings(f'{data_dir_materialproperties}/{embedding_file_materialproperties}')
node_embeddings_PFAS = load_embeddings(f'{data_dir_PFAS}/{embedding_file_PFAS}')

## Store PFAS Corpus in ChromaDB Database

In [ ]:
# Prepare PFAS chunk documents for insertion into ChromaDB.

chunks_list = sorted(glob.glob(f'{data_dir_PFAS}/*chunks_clean.csv'))

chunk_ids = []
chunks = []
titles = []

for chunks_file in chunks_list:
    try:
        print(f"Loading: {chunks_file}")
        f = pd.read_csv(chunks_file)
        title = chunks_file.replace('_chunks_clean.csv', '').split('/')[-1]
        f['title'] = title
        chunk_ids += list(f['chunk_id'])
        chunks += list(f['text'])
        titles += list(f['title'])
    except Exception as e:
        print(f"[FAIL] Could not read {chunks_file} → {e}")

PFAS_docs = [
    Document(id=id, content=chunk, metadata={"title": title})
    for id, chunk, title in zip(chunk_ids, chunks, titles)
]

# === PFAS CHECK ===
print(PFAS_docs[0].keys())
print(f"[CHECK] Loaded {len(PFAS_docs)} documents for PFAS collection.")
if len(PFAS_docs) == 0:
    print(f"[WARNING] PFAS_docs is empty. Check data_dir_PFAS or CSV contents.")
else:
    print(f" Sample PFAS doc: ID={PFAS_docs[0]['id']}, Title={PFAS_docs[0]['metadata']['title']}")
    print(PFAS_docs[0]['content'][:200] + "...")

In [ ]:
# Initialize ChromaDB persistent client and embedding function wrapper

embedding_doc_file = f'./chroma'  # Entire vector DB directory (persistent)
client = PersistentClient(path=embedding_doc_file)  # Settings telemetry left as default

In [ ]:
# Wrap ChromaDB for use with AutoGen (PFAS collection activation)

ChromaDB_PFAS = ChromaVectorDB(client=client, embedding_function=embedding_function)
ChromaDB_PFAS.active_collection = client.get_or_create_collection(
    'KG_PFASpapers', 
    embedding_function=embedding_function
)

In [ ]:
# Environment knobs: conservative batching and CUDA memory split settings.

os.environ["CHROMADB_MAX_BATCH_SIZE"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:16"

ChromaDB_PFAS = ChromaVectorDB(
    client=client,
    embedding_function=embedding_function,
)

# client.delete_collection(name='KG_PFASpapers')
ChromaDB_PFAS.active_collection = client.get_or_create_collection('KG_PFASpapers', embedding_function=embedding_function)

for i, doc in enumerate(tqdm(PFAS_docs, desc="Inserting documents into ChromaDB")):
    ChromaDB_PFAS.insert_docs(docs=[doc], collection_name='KG_PFASpapers', upsert=False)
    print(f"Inserted doc {i}: {doc['id']}")

## Wrap LLM 

In [ ]:
# Minimal OpenAI-compatible LLM wrapper for local server routing

class llm:
    def __init__(self, llm_config):
        self.client = OpenAI(api_key=llm_config["api_key"], base_url=llm_config["base_url"])
        self.model = llm_config["model"]
        self.max_tokens = llm_config["max_tokens"]
    
    def generate_cli(self, system_prompt="You are an expert in this field. Try your best to give a clear and concise answer.",
                     prompt="Hello world! I am", temperature=0):
        try:
            if system_prompt is None:
                messages = [{"role": "user", "content": prompt}]
            else:
                messages = [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": prompt},
                ]
            result = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=temperature,
                max_tokens=self.max_tokens,
            )
            return result.choices[0].message.content
        except:
            return ''

llm = llm(llm_config=config_list[0])
generate = llm.generate_cli

## Define Creative GraphWeave Agent and Hybrid GraphWeave Agent

In [ ]:
# agent definitions

from GraphReasoning import (
    extract_keywords_to_nodes,
    find_shortest_path_subgraph_between_nodes,
    convert_keywords_to_nodes,
    find_top_n_simple_paths_between_nodes,
    find_dfs_path_subgraph_between_nodes,
    bfs_with_semantic_stop,
    collect_entities,
)

logger = get_logger(__name__)

class HybridGraphWeaveAgent(UserProxyAgent):
    """(In preview) Graph Retrieval-Augmented User Proxy combining VectorDB and KG context."""
    def __init__(
        self,
        name="HybridGraphWeaveAgent",
        human_input_mode: Literal["ALWAYS", "NEVER", "TERMINATE"] = "NEVER",
        is_termination_msg: Optional[Callable[[Dict], bool]] = None,
        retrieve_config: Optional[Dict] = None,
        **kwargs,
    ):
        super().__init__(name=name, human_input_mode=human_input_mode, **kwargs)
        self._retrieve_config = {} if retrieve_config is None else retrieve_config
        self._task = self._retrieve_config.get("task", "default")
        self._vector_db = self._retrieve_config.get("vector_db", None)
        self._model = self._retrieve_config.get("model", None)
        self._max_tokens = self._retrieve_config.get("max_tokens", 8000)
        self._knowledge_graph = self._retrieve_config.get("knowledge_graph", None)
        self.customized_prompt = self._retrieve_config.get("customized_prompt", None)
        self.custom_token_count_function = self._retrieve_config.get("custom_token_count_function", count_token)
        self._context_max_tokens = self._retrieve_config.get("context_max_tokens", self._max_tokens * 0.8)
        self._n_results = self._retrieve_config.get("n_results", 3)
        self._distance_threshold = self._retrieve_config.get("distance_threshold", -1)

        self._ipython = get_ipython()
        self._results = []
        self._intermediate_answers = set()
        self._doc_contents = []
        self._doc_ids = []
        self._current_docs_in_context = []

        self.register_reply(Agent, HybridGraphWeaveAgent._generate_retrieve_user_reply, position=2)
        if not isinstance(self._vector_db, VectorDB):
            logger.error('You must provide an instance of vectordb')
            return

    def _reset(self, intermediate=False):
        self._results = []
        if not intermediate:
            self._intermediate_answers = set()
            self._doc_contents = []
            self._doc_ids = []

    def graphRAG(self, id):
        edges = list(self._knowledge_graph.out_edges(data=True))
        nodes = set()
        for edge in edges:
            if edge[2]['chunk_id'] == id:
                nodes.add(edge[0])
                nodes.add(edge[1])
        return collect_entities(self._knowledge_graph.subgraph(nodes))

    def _get_context(self):
        doc_contents = ""
        self._current_docs_in_context = []
        current_tokens = 0
        for idx, doc in enumerate(self._results[0]):
            doc = doc[0]
            graph_results = self.graphRAG(doc['id'])
            _doc_tokens = self.custom_token_count_function(doc["content"] + graph_results)
            func_print = f"Adding content of doc {doc['id']} to context from: {doc['metadata']['title']}"
            print(colored(func_print, "green"), flush=True)
            doc_contents += f"The following information related to your question is from TITLE: {' '.join(doc['metadata']['title'].split('_'))} "
            doc_contents += f"Source text: {doc['content']}\n"
            doc_contents += f"Relationships of the knowledge: {self.graphRAG(doc['id'])}\n"
            current_tokens += _doc_tokens
            func_print = f"Current tokens in use: {current_tokens}"
            print(colored(func_print, "green"), flush=True)
        return doc_contents

    def _generate_retrieve_user_reply(
        self,
        messages: Optional[List[Dict]] = None,
        sender: Optional[Agent] = None,
        config: Optional[Any] = None,
    ) -> Tuple[bool, Union[str, Dict, None]]:
        self._reset(intermediate=True)
        response = self.generate_oai_reply(messages=[self._oai_messages[sender][-1]], sender=sender)
        problems = response[1].split('\n')
        final_response = ''
        problem = problems[-1]
        final_response += f"{problem}\n"
        self.retrieve_docs(problem=problem)
        final_response += self._get_context()
        self.clear_history(sender)
        sender.clear_history(self)
        return True, final_response

    def retrieve_docs(self, problem: Union[str, List[str]] = None):
        print(colored("Retrieving for:", "green"), end=" ")
        print(f"{problem}")
        if isinstance(self._vector_db, VectorDB):
            kwargs = {}
            results = self._vector_db.retrieve_docs(
                queries=[problem],
                n_results=self._n_results,
                distance_threshold=self._distance_threshold,
                **kwargs,
            )
            self._results = results
        return

class CreativeGraphWeaveAgent(UserProxyAgent):
    def __init__(
        self,
        name="CreativeGraphWeaveAgent",
        human_input_mode: Literal["ALWAYS", "NEVER", "TERMINATE"] = "NEVER",
        is_termination_msg: Optional[Callable[[Dict], bool]] = None,
        generate=None,
        node_embeddings=None,
        embedding_tokenizer=None,
        embedding_model=None,
        retrieve_config: Optional[Dict] = None,
        use_keyword_processing: bool = False,
        **kwargs,
    ):
        super().__init__(name=name, human_input_mode=human_input_mode, **kwargs)
        self._retrieve_config = {} if retrieve_config is None else retrieve_config
        self._knowledge_graph = self._retrieve_config.get("knowledge_graph", None)
        self._n_results = self._retrieve_config.get("n_results", 5)
        self._distance_threshold = self._retrieve_config.get("distance_threshold", 0.9)
        self.algorithm = self._retrieve_config.get("algorithm", "shortest")
        self.generate = generate
        self.node_embeddings = node_embeddings
        self.embedding_tokenizer = embedding_tokenizer
        self.embedding_model = embedding_model
        self.use_keyword_processing = use_keyword_processing
        self._ipython = get_ipython()
        self._results = []
        self._intermediate_answers = set()
        self.register_reply(Agent, CreativeGraphWeaveAgent._generate_retrieve_user_reply, position=2)
        self.semanticstopnode = retrieve_config.get("semanticstopnode", None)

    def _reset(self, intermediate=False):
        self._results = []
        if not intermediate:
            self._intermediate_answers = set()
            self._doc_contents = []
            self._doc_ids = []

    def embed_text(self, text, tokenizer, model, device='cuda:0'):
        import torch
        if tokenizer:
            inputs = tokenizer(text, padding=True, truncation=True, return_tensors="pt").to(device)
            with torch.no_grad():
                outputs = model(**inputs)
            try:
                embedding = outputs.last_hidden_state.mean(dim=1).detach().cpu().numpy()
            except:
                embedding = outputs.hidden_states[-1].mean(dim=1).detach().to(torch.float).cpu().numpy()
        else:
            embedding = model.encode(text)
            if hasattr(embedding, 'cpu'):
                embedding = embedding.cpu().numpy()
        return np.asarray(embedding).reshape(1, -1)

    def _resolve_semantic_stop_node(self):
        from sklearn.metrics.pairwise import cosine_similarity
        if not hasattr(self, "semanticstopnode") or not self.semanticstopnode:
            raise ValueError("semanticstopnode must be set as a string before using bfs_semantic")
        stop_embedding = self.embed_text(self.semanticstopnode, self.embedding_tokenizer, self.embedding_model).reshape(1, -1)
        all_nodes = list(self.node_embeddings.keys())
        node_matrix = np.vstack([self.node_embeddings[node] for node in all_nodes])
        similarities = cosine_similarity(stop_embedding, node_matrix).flatten()
        best_index = similarities.argmax()
        return all_nodes[best_index]

    def process_keywords_to_graph(self, keyword_string):
        nodes = convert_keywords_to_nodes(
            keyword_string,
            self.generate,
            self.node_embeddings,
            self.embedding_tokenizer,
            self.embedding_model,
            N_samples=self._n_results,
            similarity_threshold=self._distance_threshold,
        )
        subgraph = self._select_subgraph_algorithm(nodes)
        return collect_entities(self._knowledge_graph.subgraph(subgraph))

    def graphRAG(self, message):
        if self.use_keyword_processing and isinstance(message, str) and ';' in message:
            return self.process_keywords_to_graph(message)
        else:
            nodes = extract_keywords_to_nodes(
                message,
                self.generate,
                self.node_embeddings,
                self.embedding_tokenizer,
                self.embedding_model,
                self._n_results,
                similarity_threshold=self._distance_threshold,
            )
            subgraph = self._select_subgraph_algorithm(nodes)
            return collect_entities(self._knowledge_graph.subgraph(subgraph))

    def _select_subgraph_algorithm(self, nodes):
        if self.algorithm == "shortest":
            return find_shortest_path_subgraph_between_nodes(self._knowledge_graph.to_undirected(), nodes)
        elif self.algorithm == "top_n_simple":
            return find_top_n_simple_paths_between_nodes(self._knowledge_graph.to_undirected(), nodes, n=10)
        elif self.algorithm == "dfs":
            return find_dfs_path_subgraph_between_nodes(self._knowledge_graph.to_undirected(), nodes)
        elif self.algorithm == "bfs_semantic":
            semantic_stop_node = self._resolve_semantic_stop_node()
            if semantic_stop_node not in nodes:
                nodes.append(semantic_stop_node)
            return bfs_with_semantic_stop(self._knowledge_graph.to_undirected(), nodes, semantic_stop_node=semantic_stop_node)
        else:
            raise ValueError(f"Unknown algorithm: {self.algorithm}")

    def _generate_retrieve_user_reply(
        self,
        messages: Optional[List[Dict]] = None,
        sender: Optional[Agent] = None,
        config: Optional[Any] = None,
    ) -> Tuple[bool, Union[str, Dict, None]]:
        self._reset(intermediate=True)
        if self.use_keyword_processing:
            last_eval = next(m for m in reversed(self._oai_messages[sender]) if m.get('name') == 'evaluator')
            input_message = last_eval['content']
        else:
            input_message = self._oai_messages[sender][-1]
        relationships = self.graphRAG(input_message)
        final_response = f"Please consider these knowledge graph relationships: {relationships}"
        self.clear_history(sender)
        sender.clear_history(self)
        return True, final_response

## Instantiate User and all other Agents 

In [ ]:
# Instantiate user and graph weave agents, along with planner/critic/summarizer roles.

user_proxy = autogen.UserProxyAgent(
    name="Admin",
    system_message="A human admin. Interact with the engineer to discuss the QA result.",
    human_input_mode="NEVER",
    code_execution_config=False,
    is_termination_msg=lambda x: "TERMINATE" in x.get("content", "").replace("*", "").rstrip(),
)

CreativeGraphWeaveAgent_materialproperties = CreativeGraphWeaveAgent(
    name="CreativeGraphWeaveAgent_materialproperties",
    system_message="""CreativeGraphWeaveAgent performing retrieval. 
""",
    human_input_mode="NEVER",
    code_execution_config=False,
    generate=generate,
    node_embeddings=node_embeddings_materialproperties,
    embedding_tokenizer=embedding_tokenizer,
    embedding_model=embedding_model,
    retrieve_config={
        "n_results": 5,
        "knowledge_graph": G_materialproperties,
        "distance_threshold": 1.5,
        # "algorithm": "bfs_semantic",
        "algorithm": "shortest",
        "semanticstopnode": "biocompatible"
    },
    llm_config=llm_config,
    use_keyword_processing=True,
    is_termination_msg=lambda x: "TERMINATE" in x.get("content", "").replace("*", "").rstrip(),
)

CreativeGraphWeaveAgent_PFAS = CreativeGraphWeaveAgent(
    name="CreativeGraphWeaveAgent_PFAS",
    system_message="""CreativeGraphWeaveAgent performing retrieval. 
""",
    human_input_mode="NEVER",
    code_execution_config=False,
    generate=generate,
    node_embeddings=node_embeddings_PFAS,
    embedding_tokenizer=embedding_tokenizer,
    embedding_model=embedding_model,
    retrieve_config={
        "n_results": 5,
        "knowledge_graph": G_PFAS,
        "distance_threshold": 1.5,
        # "algorithm": "bfs_semantic",
        "algorithm": "shortest",
        "semanticstopnode": "biocompatible"
    },
    llm_config=llm_config,
    use_keyword_processing=True,
    is_termination_msg=lambda x: "TERMINATE" in x.get("content", "").replace("*", "").rstrip(),
)

planner = autogen.AssistantAgent(
    name="planner",
    llm_config=llm_config,
    system_message="""Planner, you act as a supervising project manager with only a basic understanding of PFAS. 
Don't write code.

### TASK
You will:
1. FIRST create a complete numbered list of exactly 3 simple DESIGN sub-questions about how PFAS enables this application. 
 - Each sub-question must focus on only ONE intrinsic material property, factor, or requirement at a time. 
 - Sub-questions can be framed either:
 a) Directly (asking about PFAS properties), OR 
 b) Indirectly (asking about alternatives or remediation and what property gap they reveal about PFAS).
2. Then PROCEED SEQUENTIALLY through the list one by one. 
 - Only move to the next question after the previous one has been answered. 
 - Never regenerate or modify the list once it is made.
3. At the END of every question, always append a block of KEYWORDS and SYNONYMS to reinforce the context. 
 - Keywords must emphasize **specific material metrics**, **application context**, and **operational conditions**.

### RULES
1. Always number exactly 3 sub-questions upfront.
2. Do not ask multi-part or compound questions.
3. Each question must reference a measurable property and request visual proof.
4. Do not use vague or overloaded terms like "barrier properties."
5. Where corpus data emphasizes alternatives or remediation, infer the implied PFAS performance advantage and phrase the sub-question accordingly.
6. Say "WRITE REPORT" if the last question (Q3) was posed in your previous message. 
   NEVER include "WRITE REPORT" in the same message as Q3.

***
Proceed subquestion by subquestion. At the end of your response, always put the highest-priority sub-question for the agents to answer in this format: 'QUESTION: ... ?'. 
When all the subquestions have been answered, add "WRITE REPORT" at the end of your response.
""",
    is_termination_msg=lambda x: "TERMINATE" in x.get("content", "").replace("*", "").rstrip(),
)

HybridGraphWeaveAgent_PFAS = HybridGraphWeaveAgent(
    name="HybridGraphWeaveAgent_PFAS",
    system_message="""HybridGraphWeaveAgent performing retrieval. 
First look at the message you recieve and extract the question of the highest priority, often starting with QUESTION: ... , and pass it on in question-only format, such as: QUESTION: ...?
There should be always a question to consider so do not make up questions. If all the questions seem to be solved already, just answer 'TERMINATE' only.
""",
    human_input_mode="NEVER",
    code_execution_config=False,
    llm_config=llm_config,
    retrieve_config={
        "custom_token_count_function": custom_token_count_function,
        "vector_db": ChromaDB_PFAS,
        "n_results": 5,
        "max_tokens": llm_config["max_tokens"],
        "knowledge_graph": G_PFAS,
        "distance_threshold": 0.8
    },
    is_termination_msg=lambda x: "TERMINATE" in x.get("content", "").replace("*", "").rstrip(),
)

last_engineer = autogen.AssistantAgent(
    name="last_engineer",
    llm_config=llm_config,
    system_message="""Engineer with scientific backgrounds. Don't write code.
Start your response with: QUESTION: ...? \n ANSWER: ... .
You should always use the information you recieve from the other agents and don't make assumption. You should keep references when you use the provided information from another agent
Write your answer strictly in academic style with citations such as '<something true> [1]' and a references section with [1] <REFERENCE TITLE>: <reasons> and following the number in all your answers to make sure your citation is not overlapping.
Don't ever cite any sources that are not from the information you have. If you have an idea that is hypothetical, only mark it in your response.
Don't indirect cite the reference from the source texts.
Add "\nCLEAR HISTORY CreativeGraphWeaveAgent_materialproperties" at the end of your reply
""",
    is_termination_msg=lambda x: "TERMINATE" in x.get("content", "").replace("*", "").rstrip(),
)

reflective_engineer = autogen.AssistantAgent(
    name="Reflective_Engineer",
    llm_config=llm_config,
    system_message="""Engineer with scientific backgrounds. Don't write code.
Considering the crazy string of connections that the previous agent came up with, hypothesize on an idea
""",
    is_termination_msg=lambda x: "TERMINATE" in x.get("content", "").replace("*", "").rstrip(),
)

critic = autogen.AssistantAgent(
    name="Critic",
    llm_config=llm_config,
    system_message="""Senior engineer critic with semiconductor background. Don't write code.
Concisely criticize or approve whether the current sub-answer from an agent can solve the question.
Fairly evalute the completeness of the answer to the question into score on a scale of 1 to 10, 6 being acceptable. Below is the score policy you should consider.
1. Credibility: 0 to 5/5 for how much content of the answer is aquired by the information from the knowledge graphs or source texts from agents?
2. Correctness: 0 to 5/5 for how good is the answer to address the question. Does it reason the answer well?
3. Creativity: this does not give points but please report whether the answer provides any new insights based on the reasoning results.
""",
    is_termination_msg=lambda x: "TERMINATE" in x.get("content", "").replace("*", "").rstrip(),
)

summarizer = autogen.AssistantAgent(
    name="Summarizer",
    llm_config=llm_config,
    system_message="""Engineer with scientific background. Don't write code.
You write a complete academic-style report (as detailed as possible) about the formed hypothesis including the initial subquestions and subanswers discussed by all the agents that were used to understsand the design problem.
The format of your report should have all the components listed below with reorganized citations that you see only in the previous responses from the engineer agents:
1. The definition and introduction of the task
2. List all subquestions that can help solve the task and the corresponding subanswers numbering starting from a, b, c ... 
For example,
Sub-question a: ... . Sub-answer a: ...
Sub-question b: ... . Sub-answer b: ...
Sub-question c: ... . Sub-answer c: ...
3. The hypothesis generated
4. Potential issues reported
5. Summary
6. References
You only summarize the information in the chat history. Don't add new questions or assumptions. You should keep references when you use the provided information from another agent
While writing the report strictly in academic style with citations such as '<something true> [1]' and a references section with [1] <REFERENCE TITLE>: <reasons> and following the number in all your answers to make sure your citation is not overlapping.
Don't ever cite any sources that are not from the information you have. If you have an idea that is hypothetical, only mark it in your response.
If you think all the sub-questions have been handled, you should combine all the sub-questions and all the sub-answers into a concise report, keeping the original references and reasoning in the same format and numbering. No need to make too much edition. 

Add "TERMINATE" at the end of your reply
""",
    is_termination_msg=lambda x: "TERMINATE" in x.get("content", "").replace("*", "").rstrip(),
)

class Evaluator_agent(autogen.UserProxyAgent):
    def __init__(self, *args, **kwargs):
        kwargs["code_execution_config"] = False
        super().__init__(*args, **kwargs)
        self.register_reply([autogen.Agent, None], self._extract_application_keywords, position=1)
        self.design_keywords = ""
    
    def _extract_application_keywords(self, recipient, messages, sender, config):
        try:
            user_question = next(m['content'] for m in messages if m.get('name') == 'user_proxy')
        except StopIteration:
            return False, None
        retrieved_evidence = [
            f"Q: {m['content'].split('QUESTION: ')[-1]}\nA: {m['content']}\n"
            for m in messages if m.get('name') == 'HybridGraphWeaveAgent_PFAS'
        ]
        prompt = f"""ORIGINAL DESIGN QUESTION: {user_question}

 Extract the 10 most critical material parameters from this evidence:
 {"".join(retrieved_evidence)}

 STRICTLY STICK TO THIS FORMAT in your response:
 "property1 at value1; property2 in rangex to rangey; property3 at value3 ..."
 """
        reply = self.generate_reply(messages=[{"role": "user", "content": prompt}], sender=sender)
        self.design_keywords = reply.strip('"; \n')
        return True, self.design_keywords

evaluator = Evaluator_agent(
    name="evaluator",
    system_message="""Keyword extraction agent. 
Your task is to analyze design questions and RAG evidence to extract the most critical material parameters, properties, and behaviors. 

Output format requirements:
1. Strictly use: "property1 at value1; property2 in rangex to rangey; property3 at value3;..."
2. Only include parameters directly relevant to material design
3. Never include explanations or additional text
4. If no relevant parameters found, return "TERMINATE"
5. Extract as many keywords as can be gleaned from the RAG evidence that provides useful context for the application, but at a minimum extract 5 keywords
""",
    human_input_mode="NEVER",
    code_execution_config=False,
    llm_config={**llm_config, "temperature": 0.1, "max_tokens": 1000},
    is_termination_msg=lambda x: "TERMINATE" in x.get("content", "").replace("*", "").rstrip(),
)

class HypothesisMakerAgent(autogen.UserProxyAgent):
    def __init__(self, generate, *args, **kwargs):
        kwargs["code_execution_config"] = False
        super().__init__(*args, **kwargs)
        self.generate = generate
        self.hypothesis = ""
        self.register_reply([autogen.Agent, None], self._formulate_material_hypothesis, position=2)
    
    def _formulate_material_hypothesis(self, recipient, messages, sender, config):
        try:
            reasoning_path = next(m['content'] for m in reversed(messages) if m.get('name') == 'CreativeGraphWeaveAgent_materialproperties')
        except StopIteration:
            reasoning_path = "[No reasoning path found]"
        prompt = f"""
You are a materials scientist. Your task is to propose a novel, PFAS-free material or composite based on the graph-based reasoning paths provided below.

ONLY USE THIS FOLLOWING PATH AS THE SUBSTRATE for coming up with your hypothesis on a replacement to PFAS for biomedical tubing application. Ensure that you DO NOT USE any fluorine/fluoride-containing material or PFAS related materials in your hypothesis. Only suggest a reasonable hypothesis. 
## REASONING PATH (from knowledge graph):
{reasoning_path}

Your response must follow the format below. Ensure that the proposed material is formulated as a chemically and structurally integrated system—i.e., explain how the selected components are compatible and synergistically work together to fulfill the required performance criteria.

The material should be viable as a replacement for PFAS-based systems in demanding application contexts (e.g., thermal, chemical, mechanical, or biological environments). Your hypothesis must include:

- Specific expected **material property values** if available and logical from the PATH (e.g., tensile strength, Tg, gas permeability, thermal conductivity, chemical resistance, etc.)
- A rationale for how it would perform in **experimental validation tests** (e.g., freeze-thaw cycles, barrier tests, mechanical fatigue, cytotoxicity, etc.)
- A brief discussion of **why this has not yet been implemented** or may be limited in real-world deployment (e.g., manufacturing challenges, cost, scalability, regulatory uncertainty)
- A **reference to the specific knowledge graph paths** that led you to this material hypothesis (e.g., which materials or mechanisms were connected and how)

Remember: DO NOT USE any fluorine-containing material or PFAS related materials in your hypothesis. 


---
**Hypothesis:**  
A detailed paragraph proposing a PFAS-free material or composite. Include how the formulation integrates components into a stable system and enumerate expected performance metrics that match or exceed those of PFAS-based alternatives.
Only suggest a PFAS-free material or composite that could exist in the real world for the intended application. 

**Justification:**  
1. [Design requirement] → [How the material satisfies it, including quantitative performance metrics]
2. ...

**Expected Material Properties for Experimental Evaluation:**  
- Mechanical performance (e.g., tensile strength, modulus, toughness): [value and units]
- Thermal behavior (e.g., glass transition temperature Tg, melting point, thermal stability): [value and units]
- Transport properties (e.g., gas permeability, oxygen transmission rate OTR, water vapor transmission rate WVTR): [value and units]
- Chemical stability/resistance (e.g., solvents, acids, bases, oxidative environments): [value or description]
- Surface/interfacial properties (e.g., friction coefficient, wettability/contact angle, surface energy): [value and units]
- Biological compatibility (if relevant, e.g., cytotoxicity, hemocompatibility, biodegradability): [rating, description, or test outcome]


**Foreseeable Implementation Challenges:**  
List one or more challenges (e.g., dispersion uniformity, lack of scale-up protocols, cost, regulatory barriers, lack of long-term data).

**Knowledge Graph Reasoning Path(s) Used:**  
Explain which nodes, materials, or relationships from the graph were used in reasoning (e.g., "PDMS → biocompatibility + flexibility; Graphene → impermeability; Cellulose nanofiber → reinforcement synergy").
""".strip()
        try:
            response = self.generate(system_prompt="You are a materials scientist generating a new PFAS-free hypothesis based on prior design and reasoning paths.", prompt=prompt)
            self.hypothesis = response.strip()
            return True, self.hypothesis
        except Exception as e:
            print(f"[ERROR in HypothesisMakerAgent] {e}")
            return False, "TERMINATE"

hypothesis_maker = HypothesisMakerAgent(
    generate=generate,
    name="hypothesis_maker",
    system_message="You generate hypotheses for PFAS alternatives based on design needs and reasoning graphs.",
    human_input_mode="NEVER",
    llm_config=llm_config,
    is_termination_msg=lambda x: "TERMINATE" in x.get("content", "").replace("*", "").rstrip(),
)

## Define Speaker Selection

In [ ]:
agents = [user_proxy, planner, HybridGraphWeaveAgent_PFAS, summarizer, evaluator, CreativeGraphWeaveAgent_materialproperties, hypothesis_maker]

def graph_speaker_selection_func(last_speaker: Agent, groupchat: autogen.GroupChat):
    """Optimized speaker selection without manual message passing."""
    messages = groupchat.messages
    if not hasattr(groupchat, 'subquestions'):
        groupchat.subquestions = {'original_list': None, 'current_index': 0, 'answered': []}
    if last_speaker is user_proxy:
        return planner
    if "WRITE REPORT" in messages[-1]["content"].upper():
        return summarizer
    if last_speaker is planner:
        if groupchat.subquestions['original_list'] is None:
            content = messages[-1]["content"]
            questions = [line[line.find(' ')+1:].strip() for line in content.split('\n') if line.strip().startswith(('1. ', '2. ', '3. ', '4. '))]
            groupchat.subquestions['original_list'] = questions
        return HybridGraphWeaveAgent_PFAS
    if last_speaker is HybridGraphWeaveAgent_PFAS:
        groupchat.subquestions['current_index'] += 1
        return evaluator if groupchat.subquestions['current_index'] >= len(groupchat.subquestions['original_list']) else planner
    if last_speaker is evaluator:
        return CreativeGraphWeaveAgent_materialproperties
    if last_speaker is CreativeGraphWeaveAgent_materialproperties:
        groupchat.subquestions = {'original_list': None, 'current_index': 0, 'answered': []}
        return hypothesis_maker
    return "auto"

groupchat = autogen.GroupChat(
    agents=agents,
    messages=[],
    max_round=100,
    speaker_selection_method=graph_speaker_selection_func,
    enable_clear_history=True,
)
manager = autogen.GroupChatManager(groupchat)

## Clear Cache

In [ ]:
# Clean any residual cache between runs

try:
    shutil.rmtree('.cache')
except:
    pass

## Set the User Query

In [ ]:
# Define the query set for experiments

q = []
q.append('''
What are the specific material properties of PFAS that make it used for the application of biomedical tubing? 
''')


## Start Conversation

In [ ]:
# Run the group conversation for each input question. Results are accumulated.

result_experiments = []
for q_ in q:
    result_experiments.append(
        user_proxy.initiate_chat(
            manager,
            message=f'''{q_}\n''',
        )
    )

## Optionally Store Results

In [ ]:
# Persist responses into a simple text artifact

with open("shortest.txt", "w") as text_file:
    for q_, r in zip(q, result_experiments):
        text_file.write(f"Question: {q_}\nResponse: {r}\n\n")

In [ ]:
# Filter the current notebook to only the cell(s) containing the experiment snippet and export to HTML.

notebook_path = "graphAgents.ipynb"
search_snippet = "result_experiments = []"
nb = nbformat.read(notebook_path, as_version=4)
matched_cells = [cell for cell in nb.cells if cell.cell_type == "code" and search_snippet in cell.source]
if not matched_cells:
    raise ValueError(f"No cell found containing: {search_snippet}")
nb.cells = matched_cells
exporter = HTMLExporter(template_name="lab")
(body, resources) = exporter.from_notebook_node(nb)
out_html = "shortest.html"
with open(out_html, "w", encoding="utf-8") as f:
    f.write(body)
print(f"Saved filtered notebook cell to {out_html}")